# CXR Insight — fine-tune a small vision-language model on free Colab

Research prototype; not for clinical use. Fine-tunes a LoRA adapter for report
generation with Unsloth, saving checkpoints to Google Drive and resuming after a
disconnect. Adapted from Unsloth's vision fine-tuning notebooks
(https://github.com/unslothai/notebooks); keep that attribution.

**Data you must supply** (not included in the repository): a JSONL file per split
(`train.jsonl`, `validation.jsonl`, `test.jsonl`) under `DATA`, each row
`{"id", "image_path", "report", "patient_id", "report_source"}`, split by patient,
with documented source and usage terms. Do **not** use the legacy synthetic CSV: all of
its reports carry a chat-role artefact and most copy the prompt's example
(see docs/PROVENANCE.md).

Runtime → Change runtime type → T4 GPU (free tier).

In [ ]:
!pip install -q unsloth
!git clone https://github.com/devbyabir/cxr-insight.git
%cd cxr-insight
!pip install -q -e .

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/cxr-insight/report_data"      # your prepared splits + images
RUN = "/content/drive/MyDrive/cxr-insight/runs/vlm_seed3407"  # checkpoints persist here
from cxr_insight.config import load_config
cfg = load_config("configs/vlm_qwen2vl_2b.toml")   # model + training settings (versioned)
MODEL_ID, REVISION = cfg.model.base_model_id, cfg.model.revision  # pin revision for reported runs
T = cfg.training
SEED, MAX_STEPS, SAVE_STEPS = T.seed, 300, 50
from cxr_insight.inference import DEFAULT_INSTRUCTION as INSTRUCTION  # shared with inference code

In [ ]:
import json, os
from PIL import Image
def read(split):
    with open(f"{DATA}/{split}.jsonl") as f:
        return [json.loads(l) for l in f if l.strip()]
train, val, test = read("train"), read("validation"), read("test")
for a, b in [(train, val), (train, test), (val, test)]:
    assert not {r["patient_id"] for r in a} & {r["patient_id"] for r in b}, "patient overlap"
def to_conversation(row):
    image = Image.open(os.path.join(DATA, row["image_path"])).convert("RGB")
    return {"messages": [
        {"role": "user", "content": [{"type": "image", "image": image},
                                     {"type": "text", "text": INSTRUCTION}]},
        {"role": "assistant", "content": [{"type": "text", "text": row["report"]}]}]}
train_ds = [to_conversation(r) for r in train]
print(len(train), len(val), len(test))

In [ ]:
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
model, processor = FastVisionModel.from_pretrained(MODEL_ID, load_in_4bit=True,
    use_gradient_checkpointing="unsloth", **({"revision": REVISION} if REVISION else {}))
# Training is the ONLY place a new adapter is attached.
model = FastVisionModel.get_peft_model(model, finetune_vision_layers=False,
    finetune_language_layers=True, finetune_attention_modules=True, finetune_mlp_modules=True,
    r=T.lora_r, lora_alpha=T.lora_alpha, lora_dropout=0, bias="none", random_state=SEED)
FastVisionModel.for_training(model)
trainer = SFTTrainer(model=model, tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor), train_dataset=train_ds,
    args=SFTConfig(per_device_train_batch_size=T.batch_size,
        gradient_accumulation_steps=T.gradient_accumulation_steps,
        warmup_steps=10, max_steps=MAX_STEPS, learning_rate=T.learning_rate, optim="adamw_8bit",
        weight_decay=0.01, lr_scheduler_type="linear", seed=SEED, logging_steps=10,
        output_dir=f"{RUN}/checkpoints", save_steps=SAVE_STEPS, save_total_limit=2,
        report_to="none", remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True}, max_length=T.max_seq_length))
has_ckpt = os.path.isdir(f"{RUN}/checkpoints") and any(
    d.startswith("checkpoint-") for d in os.listdir(f"{RUN}/checkpoints"))
trainer.train(resume_from_checkpoint=has_ckpt)   # resumes after a disconnect
model.save_pretrained(f"{RUN}/adapter"); processor.save_pretrained(f"{RUN}/adapter")
from cxr_insight.runinfo import run_metadata
json.dump(run_metadata(SEED, {"config": cfg.to_dict(), "model_id": MODEL_ID, "revision": REVISION, "max_steps": MAX_STEPS,
          "train_cases": len(train), "data_dir": DATA}), open(f"{RUN}/run_metadata.json", "w"), indent=2)

Runtime → Restart session, then run the setup cells (not training) and continue.

In [ ]:
# Verify the TRAINED adapter really loads (non-zero LoRA weights), then generate.
import json, gc, torch
from cxr_insight.models import load_report_model, verify_trained_adapter
from cxr_insight.inference import generate_text, load_image, GenerationSettings
settings = GenerationSettings(do_sample=False, temperature=0, max_new_tokens=256)
tuned = load_report_model(MODEL_ID, REVISION, adapter_path=f"{RUN}/adapter")
print(verify_trained_adapter(tuned.model))
def run(rm, path):
    with open(path, "w") as f:
        for r in test:
            text = generate_text(rm, load_image(os.path.join(DATA, r["image_path"])), INSTRUCTION, settings)
            f.write(json.dumps({"id": r["id"], "reference": r["report"], "generated": text}) + "\n")
run(tuned, f"{RUN}/test_finetuned.jsonl")
del tuned; gc.collect(); torch.cuda.empty_cache()
base = load_report_model(MODEL_ID, REVISION)          # same cases, base model
run(base, f"{RUN}/test_baseline.jsonl")

In [ ]:
!python -m cxr_insight.report_eval baseline="$RUN/test_baseline.jsonl" finetuned="$RUN/test_finetuned.jsonl" --out "$RUN/report_comparison.json"